In [ ]:
from dnallm import load_config, load_model_and_tokenizer, DNATrainer
from dnallm.datahandling import show_preset_dataset, load_preset_dataset
from torch.nn.init import xavier_uniform_

### Finetune

In [ ]:
# Load the config file
configs = load_config("./finetune_config.yaml")
# Task configs
configs["task"].task_type = "multilabel"
configs["task"].label_names = [
    "ATAC_7days_leaf_rep1",
    "ATAC_7days_leaf_rep2",
    "ATAC_mesophyll_cell_rep1",
    "ATAC_mesophyll_cell_rep2",
    "ATAC_mesophyll_cell_rep3",
    "ATAC_root_hair_rep1",
    "ATAC_root_hair_rep2",
    "ATAC_root_non_hair_rep1",
    "ATAC_root_non_hair_rep2",
    "ATAC_root_tip_rep1",
    "ATAC_root_tip_rep2",
    "ATAC_stem_cell_rep1",
    "ATAC_stem_cell_rep2",
    "ATAC_stem_cell_rep3",
    "DNase_flower_14_days",
    "DNase_inflorescence_normal",
    "DNase_open_flower_normal",
    "DNase_root_7_days",
    "DNase_seedling_normal"
]
configs["task"].num_labels = len(configs["task"].label_names)
# Finetune configs
configs["finetune"].per_device_train_batch_size = 24
configs["finetune"].per_device_eval_batch_size = 48
configs["finetune"].logging_steps = 10000
configs["finetune"].eval_steps = 10000
configs["finetune"].save_steps = 10000
configs["finetune"].num_train_epochs = 1
configs["finetune"].metric_for_best_model = "f1"
configs["finetune"].output_dir = "outputs_multi_label"
configs["finetune"].bf16 = True

In [ ]:
# Load the model and tokenizer
model_name = "zhangtaolab/plant-dnamodernbert-BPE"
# from ModelScope
model, tokenizer = load_model_and_tokenizer(model_name, task_config=configs["task"], source="modelscope")

In [ ]:
dataset_name = "plant-genomic-benchmark"
PLANT_DATASET = show_preset_dataset()[dataset_name]
print(PLANT_DATASET["tasks"])

In [ ]:
# Load a preset dataset
task = "chromatin_access.arabidopis_thaliana"
datasets = load_preset_dataset(dataset_name=dataset_name, task=task)

# Format labels
for split in datasets.dataset:
    datasets.dataset[split] = datasets._format_labels(
        datasets.dataset[split],
        multi_label_sep=datasets.multi_label_sep
    )

# Tokenize the dataset
datasets.encode_sequences(tokenizer=tokenizer)

In [ ]:
# Initialize the trainer
xavier_uniform_(model.classifier.weight)
trainer = DNATrainer(
    model=model,
    config=configs,
    datasets=datasets
)

In [ ]:
# Start training
metrics = trainer.train()
print(metrics)

In [ ]:
# Plot the radar chart
from dnallm.inference.plot import plot_radar

In [ ]:
chart = plot_radar(metrics, metric="AUPRC")
chart